# COS40007 Artificial Intelligence Engineering
# Portfolio Assessment 2: Advanced Data Preprocessing, Feature Engineering & Model Development

**Unit:** COS40007 Artificial Intelligence Engineering  
**Assessment:** Portfolio Assessment 2  
**Student Name:** Vu Ba Hien  
**Student ID:** 104995634  
**Dataset:** Concrete Compressive Strength (UCI Machine Learning Repository / Yeh, 1998)  

---

## Acknowledgement of Country
> *"We respectfully acknowledge the Wurundjeri Woi-wurrung people of the Kulin Nation, the Traditional Custodians of the lands upon which Swinburne University of Technology campuses are located, and pay our respects to their Elders past, present, and emerging. We extend that respect to all Aboriginal and Torres Strait Islander peoples and recognise their enduring connection to country, knowledge systems, and culture."*

---

## Generative AI Use Declaration
> *"I used Google Gemini (Antigravity AI) to assist with checking code syntax, commenting, debugging, and discussing domain feature ideas (like Abrams' water-cement ratio and log curing age). All final code, interpretations, model evaluations, and written reflections were reviewed, verified, and completed by myself."*


## 0. Setup and Environment
First, we import the core scientific libraries (`numpy`, `pandas`), visualization tools (`matplotlib`, `seaborn`), and the machine learning modules from `scikit-learn` for preprocessing, Decision Trees, Support Vector Machines, cross-validation, and metrics.


In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.svm import SVC
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support

# Setup clean visual styling
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 10
RANDOM_STATE = 42

print("All libraries imported successfully.")


## Task 1: Target Variable Preparation
### 1.1 Target Discretisation
In our Task 1 EDA, concrete compressive strength (`strength`) was a continuous measurement ranging between **2.33 MPa and 82.60 MPa** (Mean = 35.82 MPa, Std = 16.71 MPa).

In civil engineering (specifically **Australian Standard AS 3600**), engineers categorize concrete by strength grades depending on its intended structural use:
1. **Low Strength ($\le 26.97$ MPa)**: Non-structural blinding, residential pavements, mass footings ($\le$ 25 MPa).
2. **Medium Strength ($26.97$ to $41.37$ MPa)**: Standard commercial building structural members: beams, slabs, columns (25–40 MPa).
3. **High Strength ($> 41.37$ MPa)**: High-rise columns, prestressed bridge girders, heavy infrastructure (> 40 MPa).

We used **equal-frequency tertile binning (`pd.qcut`)** to turn this continuous value into **3 balanced classes**. Each category contains roughly one-third of our 1,030 samples (~33.3% each). This balanced split is essential because it stops our models from picking up a majority-class bias.


In [2]:
# Load the dataset
df = pd.read_csv('data/concrete_data.csv')

# Split target into 3 balanced categories using quantiles
labels = ['Low Strength', 'Medium Strength', 'High Strength']
df['target_class'], bins = pd.qcut(df['strength'], q=3, labels=[0, 1, 2], retbins=True)
df['target_label'] = pd.qcut(df['strength'], q=3, labels=labels)

print(f"Target Threshold Bins: {bins}")
print("\nClass Distribution:")
display(df['target_label'].value_counts())

# Plot the balanced target distribution
fig, ax = plt.subplots(figsize=(7, 4), dpi=150)
counts = df['target_label'].value_counts()[labels]
palette = ['#3498db', '#2ecc71', '#e74c3c']
bars = ax.bar(labels, counts, color=palette, edgecolor='black', width=0.5)
for bar in bars:
    y = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, y + 5, f"{y} ({y/len(df)*100:.1f}%)", ha='center', weight='bold')
ax.set_title("Discretised Target Class Distribution (3 Balanced Classes)", weight='bold')
ax.set_ylabel("Number of Samples")
ax.set_ylim(0, max(counts)*1.15)
plt.tight_layout()
plt.show()


## Task 2: Feature Normalisation & Transformation
### 2.1 Distribution Analysis & Justification for Scaling
Looking back at our EDA plots, the 8 features had very different scales and distribution shapes:
- **`coarseagg` & `fineagg` (Skewness: -0.04 and -0.25):** Both show clean, symmetric bell curves. We scaled them with **StandardScaler** to give them zero mean and unit variance.
- **`cement` & `water` (Skewness: 0.50 and 0.08):** Fairly evenly spread physical quantities. We also scaled them with **StandardScaler**.
- **`slag`, `ash`, and `superplastic` (Skewness: 0.87, 0.54, 0.91):** These contain a lot of zeros (between 36% and 55% of the data) because many standard batches don't use chemical or mineral admixtures. We used **RobustScaler** here because it scales using the median and interquartile range (IQR), so the heavy zero cluster doesn't distort the scale.
- **`age` (Skewness: 3.27):** Very heavily right-skewed. Half the concrete tests happen at or before 28 days, but some long tests extend out to 365 days. We used **RobustScaler** to prevent those 365-day points from dominating the distance calculations.

### 2.2 Feature Encoding
All 8 original predictors are physical quantities (mass in kg/m³ and curing age in days), so they didn't require one-hot or ordinal encoding. Our discretised target categories were encoded as integers (`0`, `1`, `2`).

### 2.3 How Scaling Impacts Different Model Types
- **Decision Trees:** Decision trees make splits by checking whether a single feature is above or below a threshold ($x_j \le 	ext{value}$). Monotonic scaling changes the numbers but keeps their exact relative order the same. Because of this, **Decision Trees are scale-invariant**—scaling won't change their splits or performance.
- **Support Vector Machines (SVM):** SVM with an RBF kernel relies directly on Euclidean distance:
  $$K(\mathbf{x}_i, \mathbf{x}_j) = \exp\left(-\gamma \|\mathbf{x}_i - \mathbf{x}_j\|^2ight)$$
  Without scaling, coarse aggregate (values around 1000) would completely overpower superplasticizer (values around 6) in every distance calculation. For SVM, **normalisation is strictly required**.


In [3]:
raw_features = ['cement', 'slag', 'ash', 'water', 'superplastic', 'coarseagg', 'fineagg', 'age']

scaler_std = StandardScaler()
scaler_rob = RobustScaler()

df_scaled = df.copy()
standard_cols = ['cement', 'water', 'coarseagg', 'fineagg']
robust_cols = ['slag', 'ash', 'superplastic', 'age']

df_scaled[standard_cols] = scaler_std.fit_transform(df[standard_cols])
df_scaled[robust_cols] = scaler_rob.fit_transform(df[robust_cols])

# Compare distributions before and after scaling
fig, axes = plt.subplots(2, 4, figsize=(16, 7), dpi=150)
axes = axes.flatten()
for idx, col in enumerate(raw_features):
    sns.kdeplot(df[col], ax=axes[idx], color='#2980b9', fill=True, alpha=0.3)
    axes[idx].set_title(f"{col} (Raw, Skew: {df[col].skew():.2f})", weight='bold')
plt.suptitle("Raw Feature Distributions Before Normalisation", y=1.01, fontsize=14, weight='bold')
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 4, figsize=(16, 7), dpi=150)
axes = axes.flatten()
for idx, col in enumerate(raw_features):
    sns.kdeplot(df_scaled[col], ax=axes[idx], color='#27ae60', fill=True, alpha=0.3)
    axes[idx].set_title(f"{col} (Scaled)", weight='bold')
plt.suptitle("Feature Distributions After Normalisation", y=1.01, fontsize=14, weight='bold')
plt.tight_layout()
plt.show()


## Task 3: Feature Engineering
Using what we learned from our EDA and principles from concrete materials engineering, we created **4 new domain features**:

1. **Water-to-Binder Ratio (`w_b_ratio`)**:
   $$	ext{w\_b\_ratio} = rac{	ext{water}}{	ext{cement} + 	ext{slag} + 	ext{ash} + 10^{-6}}$$
   - *EDA Motivation:* In EDA, water had a negative correlation with strength ($r = -0.29$) while cement ($r = +0.50$) and slag ($r = +0.13$) were positive.
   - *Engineering Meaning:* This is **Abrams' Law (1918)**—the most fundamental law of concrete technology. Concrete compressive strength is governed by the ratio of water to cementitious material, because excess water creates microscopic pores as it evaporates, weakening the hardened paste.
   - *Validation:* It produced a correlation of **$r = -0.623$** with compressive strength—stronger than any individual raw feature in the dataset!

2. **Total Binder Content (`total_binder`)**:
   $$	ext{total\_binder} = 	ext{cement} + 	ext{slag} + 	ext{ash}$$
   - *EDA Motivation:* Concrete mixes often substitute some Portland cement with slag or fly ash.
   - *Engineering Meaning:* Summing them up gives the total reactive cementitious mass available to form C-S-H gel.
   - *Validation:* Strong positive correlation with strength of **$r = +0.613$**.

3. **Aggregate-to-Binder Ratio (`agg_binder_ratio`)**:
   $$	ext{agg\_binder\_ratio} = rac{	ext{coarseagg} + 	ext{fineagg}}{	ext{total\_binder} + 10^{-6}}$$
   - *EDA Motivation:* Coarse and fine aggregates both correlated negatively with strength ($r pprox -0.16$).
   - *Engineering Meaning:* This represents the ratio of aggregate rocks and sand to cement paste. If there is too much aggregate, there isn't enough paste to coat the particles, creating weak interfacial transition zones (ITZ).
   - *Validation:* Strong negative correlation with strength of **$r = -0.566$**.

4. **Log Curing Age (`log_age`)**:
   $$	ext{log\_age} = \ln(	ext{age} + 1)$$
   - *EDA Motivation:* Age was heavily right-skewed (skewness 3.27), and scatter plots showed that strength gains plateau after 28 days.
   - *Engineering Meaning:* Cement hydration reaction kinetics slow down logarithmically over time.
   - *Validation:* Taking the log boosted correlation from $r = +0.329$ (raw age) to **$r = +0.549$**!


In [4]:
# Compute our 4 domain features
df['total_binder'] = df['cement'] + df['slag'] + df['ash']
df['w_b_ratio'] = df['water'] / (df['total_binder'] + 1e-6)
df['agg_binder_ratio'] = (df['coarseagg'] + df['fineagg']) / (df['total_binder'] + 1e-6)
df['log_age'] = np.log1p(df['age'])

engineered_features = ['total_binder', 'w_b_ratio', 'agg_binder_ratio', 'log_age']

print("--- Pearson Correlations of Engineered Features with Compressive Strength ---")
display(df[engineered_features + ['strength']].corr()['strength'].drop('strength'))

# Plot relationships with target
fig, axes = plt.subplots(2, 2, figsize=(12, 9), dpi=150)
sns.scatterplot(data=df, x='w_b_ratio', y='strength', hue='target_label', palette=palette, ax=axes[0,0], alpha=0.6)
axes[0,0].set_title("Water-to-Binder Ratio vs Strength (r = -0.62)", weight='bold')

sns.scatterplot(data=df, x='total_binder', y='strength', hue='target_label', palette=palette, ax=axes[0,1], alpha=0.6)
axes[0,1].set_title("Total Binder vs Strength (r = +0.61)", weight='bold')

sns.scatterplot(data=df, x='agg_binder_ratio', y='strength', hue='target_label', palette=palette, ax=axes[1,0], alpha=0.6)
axes[1,0].set_title("Aggregate-to-Binder Ratio vs Strength (r = -0.57)", weight='bold')

sns.scatterplot(data=df, x='log_age', y='strength', hue='target_label', palette=palette, ax=axes[1,1], alpha=0.6)
axes[1,1].set_title("Log Curing Age vs Strength (r = +0.55)", weight='bold')

plt.suptitle("Validation of Engineered Features Against Compressive Strength", y=1.01, fontsize=14, weight='bold')
plt.tight_layout()
plt.show()


## Task 4 & Task 5: Decision Tree Modelling & First Comparison Table
We tested Decision Tree Classifiers (`max_depth=5`, `random_state=42`) across **5 different feature sets**:
- **Set 1: Original Features (8 features)**: All 8 raw physical predictors.
- **Set 2: Top Correlated Features (4 features)**: `cement`, `superplastic`, `age`, `water`.
- **Set 3: Original + Engineered Features (12 features)**: All 8 raw features plus our 4 domain features.
- **Set 4: PCA Features (5 components)**: The top 5 principal components.
- **Set 5: Domain Minimal Set (4 features)**: Core Abrams' Law features (`total_binder`, `w_b_ratio`, `log_age`, `superplastic`).

We used a **70% training and 30% testing stratified split** (`random_state=42`), and ran **5-Fold Cross-Validation** on the training data.


In [5]:
all_features = raw_features + engineered_features
X_all = pd.DataFrame(StandardScaler().fit_transform(df[all_features]), columns=all_features)
y = df['target_class'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Prepare PCA for Set 4
pca_dt = PCA(n_components=5, random_state=RANDOM_STATE)
X_train_pca = pca_dt.fit_transform(X_train[raw_features])
X_test_pca = pca_dt.transform(X_test[raw_features])

feature_sets = {
    'Set 1: Original Features': (X_train[raw_features], X_test[raw_features], 8, 'All 8 original predictors'),
    'Set 2: Top Correlated Features': (X_train[['cement', 'superplastic', 'age', 'water']], X_test[['cement', 'superplastic', 'age', 'water']], 4, 'Top 4 EDA correlated features'),
    'Set 3: Original + Engineered': (X_train[all_features], X_test[all_features], 12, '8 raw + 4 engineered features'),
    'Set 4: PCA Features': (X_train_pca, X_test_pca, 5, 'First 5 principal components'),
    'Set 5: Domain Minimal Set': (X_train[['total_binder', 'w_b_ratio', 'log_age', 'superplastic']], X_test[['total_binder', 'w_b_ratio', 'log_age', 'superplastic']], 4, 'Core Abrams law features')
}

dt_summary = []
trained_trees = {}

for name, (xtr, xte, nfeat, desc) in feature_sets.items():
    clf = DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE)
    clf.fit(xtr, y_train)
    trained_trees[name] = clf
    
    tr_acc = accuracy_score(y_train, clf.predict(xtr))
    te_acc = accuracy_score(y_test, clf.predict(xte))
    cv_scores = cross_val_score(clf, xtr, y_train, cv=cv, scoring='accuracy')
    
    dt_summary.append({
        'Feature Set': name,
        'Features': nfeat,
        'Description': desc,
        'Train Acc (%)': tr_acc * 100,
        'Test Acc (%)': te_acc * 100,
        '5-Fold CV Mean (%)': cv_scores.mean() * 100,
        '5-Fold CV Std (%)': cv_scores.std() * 100
    })

dt_table_df = pd.DataFrame(dt_summary)
print("=== First Summary Table: Decision Tree Comparison Across 5 Feature Sets ===")
display(dt_table_df)

# Plot comparison bar chart
fig, ax = plt.subplots(figsize=(10, 5), dpi=150)
x_pos = np.arange(len(dt_table_df))
width = 0.35
ax.bar(x_pos - width/2, dt_table_df['Test Acc (%)'], width, label='Test Accuracy (70/30 Split)', color='#3498db', edgecolor='black')
ax.bar(x_pos + width/2, dt_table_df['5-Fold CV Mean (%)'], width, yerr=dt_table_df['5-Fold CV Std (%)'],
       capsize=5, label='5-Fold CV Accuracy (Mean ± Std)', color='#2ecc71', edgecolor='black')

ax.set_xticks(x_pos)
ax.set_xticklabels([f"Set {i+1}\n({r['Features']} feats)" for i, r in enumerate(dt_summary)])
ax.set_ylabel("Accuracy (%)", weight='bold')
ax.set_title("Decision Tree Performance Across 5 Feature Sets (First Comparison Table)", weight='bold')
ax.set_ylim(45, 95)
ax.legend()
plt.tight_layout()
plt.show()

# Best Decision Tree Feature Importance
best_tree = trained_trees['Set 3: Original + Engineered']
fi_df = pd.DataFrame({'Feature': all_features, 'Importance': best_tree.feature_importances_}).sort_values('Importance', ascending=False)
fig, ax = plt.subplots(figsize=(8, 4.5), dpi=150)
sns.barplot(data=fi_df, x='Importance', y='Feature', color='#2980b9', ax=ax, edgecolor='black')
ax.set_title("Feature Importances of Best Decision Tree (Set 3: Original + Engineered)", weight='bold')
plt.tight_layout()
plt.show()


### Reflections on Decision Tree Performance:
1. **Which feature set performed best and why?**
   - **Set 3 (Original + Engineered Features)** had the highest cross-validation score (**77.54% ± 1.73%**), closely followed by **Set 5 (Domain Minimal Set)** with **76.29% ± 3.15%**. Both beat the original features (Set 1: 70.32%) by more than **+7%**.
   - *Why?* Decision trees split along horizontal and vertical lines. Because physical relationships like the water-cement ratio and curing curves are non-linear, providing them pre-calculated makes it much easier for the tree to make clean, accurate splits without having to build a messy, overfitted staircase.
2. **Why did PCA (Set 4) perform so poorly on Decision Trees?**
   - Set 4 collapsed to **54.37% CV accuracy** (barely better than a coin flip). PCA blends features into linear combinations, which rotates the coordinate axes. Trying to split rotated data using axis-aligned orthogonal cuts forces the decision tree to make lots of tiny fragmented steps, leading to severe underfitting.
3. **Takeaway from Minimal Set (Set 5):**
   - Using just **4 domain features** (`total_binder`, `w_b_ratio`, `log_age`, `superplastic`), Set 5 scored **76.29% CV accuracy**, comfortably beating the 8 raw features of Set 1. This shows that understanding the domain physics lets us build much simpler, more interpretable models.


## Task 6: Model Improvement with Support Vector Classifier (SVM)
Decision trees only create blocky step functions, but concrete hydration happens continuously as crystals grow. To improve on this, we implemented a **Support Vector Classifier (SVC)** using a Radial Basis Function (RBF) kernel on our scaled features.


In [6]:
svm_baseline = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=RANDOM_STATE)
svm_baseline.fit(X_train[all_features], y_train)

svm_base_tr = accuracy_score(y_train, svm_baseline.predict(X_train[all_features]))
svm_base_te = accuracy_score(y_test, svm_baseline.predict(X_test[all_features]))
svm_base_cv = cross_val_score(svm_baseline, X_train[all_features], y_train, cv=cv, scoring='accuracy')

print(f"--- Baseline SVM (RBF, default C=1.0, gamma='scale') ---")
print(f"Train Accuracy:     {svm_base_tr*100:.2f}%")
print(f"Test Accuracy:      {svm_base_te*100:.2f}%")
print(f"5-Fold CV Accuracy: {svm_base_cv.mean()*100:.2f}% ± {svm_base_cv.std()*100:.2f}%")


## Task 7: Hyperparameter Tuning via GridSearchCV
Next, we ran a systematic `GridSearchCV` over the main SVM hyperparameters:
- Regularization parameter $C \in [0.1, 1.0, 10.0, 50.0, 100.0]$
- Kernel coefficient $\gamma \in [	ext{'scale'}, 	ext{'auto'}, 0.01, 0.05, 0.1, 0.5]$
- Kernel function $\in [	ext{'rbf'}, 	ext{'linear'}, 	ext{'poly'}]$


In [7]:
param_grid = {
    'C': [0.1, 1.0, 10.0, 50.0, 100.0],
    'gamma': ['scale', 'auto', 0.01, 0.05, 0.1, 0.5],
    'kernel': ['rbf', 'linear', 'poly']
}

grid_search = GridSearchCV(
    estimator=SVC(random_state=RANDOM_STATE),
    param_grid=param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1
)
grid_search.fit(X_train[all_features], y_train)

best_svm = grid_search.best_estimator_
svm_tuned_tr = accuracy_score(y_train, best_svm.predict(X_train[all_features]))
svm_tuned_te = accuracy_score(y_test, best_svm.predict(X_test[all_features]))
svm_tuned_cv = cross_val_score(best_svm, X_train[all_features], y_train, cv=cv, scoring='accuracy')

print(f"Best Hyperparameters: {grid_search.best_params_}")
print(f"Tuned Test Accuracy:      {svm_tuned_te*100:.2f}%")
print(f"Tuned 5-Fold CV Accuracy: {svm_tuned_cv.mean()*100:.2f}% ± {svm_tuned_cv.std()*100:.2f}%")
print(f"Absolute Gain over Baseline SVM: +{(svm_tuned_cv.mean() - svm_base_cv.mean())*100:.2f}%")


## Task 8: Feature Selection with SelectKBest
We tested feature selection using `SelectKBest` with ANOVA F-tests (`f_classif`) across $K=2$ to $12$ features to see if dropping weaker predictors would help the model generalize better.


In [8]:
k_range = list(range(2, len(all_features) + 1))
k_scores = []

for k in k_range:
    sel = SelectKBest(score_func=f_classif, k=k)
    X_tr_k = sel.fit_transform(X_train[all_features], y_train)
    clf_k = SVC(C=best_svm.C, gamma=best_svm.gamma, kernel=best_svm.kernel, random_state=RANDOM_STATE)
    score = cross_val_score(clf_k, X_tr_k, y_train, cv=cv, scoring='accuracy').mean()
    k_scores.append(score)

best_k = k_range[np.argmax(k_scores)]

# Plot K vs CV Accuracy
fig, ax = plt.subplots(figsize=(8, 4), dpi=150)
ax.plot(k_range, np.array(k_scores)*100, marker='o', color='#8e44ad', linewidth=2)
ax.axvline(best_k, color='#e74c3c', linestyle='--', label=f'Optimal K = {best_k}')
ax.set_title("SVM 5-Fold CV Accuracy vs Number of Selected Features (K)", weight='bold')
ax.set_xlabel("Number of Features (K)")
ax.set_ylabel("5-Fold CV Accuracy (%)")
ax.legend()
plt.tight_layout()
plt.show()

# Fit optimal selector with best K
opt_sel = SelectKBest(score_func=f_classif, k=best_k)
X_tr_kbest = opt_sel.fit_transform(X_train[all_features], y_train)
X_te_kbest = opt_sel.transform(X_test[all_features])
selected_feats = [feat for feat, s in zip(all_features, opt_sel.get_support()) if s]
print(f"Optimal K = {best_k} Features Selected: {selected_feats}")

svm_kbest = SVC(C=best_svm.C, gamma=best_svm.gamma, kernel=best_svm.kernel, random_state=RANDOM_STATE)
svm_kbest.fit(X_tr_kbest, y_train)

svm_kbest_tr = accuracy_score(y_train, svm_kbest.predict(X_tr_kbest))
svm_kbest_te = accuracy_score(y_test, svm_kbest.predict(X_te_kbest))
svm_kbest_cv = cross_val_score(svm_kbest, X_tr_kbest, y_train, cv=cv, scoring='accuracy')

print(f"SVM + K-Best (K={best_k}) Test Acc: {svm_kbest_te*100:.2f}%")
print(f"SVM + K-Best (K={best_k}) CV Mean: {svm_kbest_cv.mean()*100:.2f}% ± {svm_kbest_cv.std()*100:.2f}%")


## Task 9: Dimensionality Reduction with PCA
We ran PCA on our scaled 12 features to see how much we could compress the input dimensions while retaining at least 90% of the total variance.


In [9]:
pca = PCA(random_state=RANDOM_STATE)
pca.fit(X_train[all_features])

cum_var = np.cumsum(pca.explained_variance_ratio_)
n_components_90 = int(np.argmax(cum_var >= 0.90) + 1)
print(f"Optimal Components for >= 90% Variance: {n_components_90} (Preserves {cum_var[n_components_90-1]*100:.2f}% Variance)")

# Scree Plot
fig, ax1 = plt.subplots(figsize=(9, 4.5), dpi=150)
x_comps = range(1, len(cum_var) + 1)
ax1.bar(x_comps, pca.explained_variance_ratio_*100, alpha=0.6, color='#3498db', edgecolor='black', label='Individual Variance (%)')
ax1.set_xlabel('Principal Component')
ax1.set_ylabel('Individual Explained Variance (%)', color='#2980b9')

ax2 = ax1.twinx()
ax2.plot(x_comps, cum_var*100, marker='s', color='#e67e22', linewidth=2.5, label='Cumulative Variance (%)')
ax2.axhline(90, color='#e74c3c', linestyle='--', label='90% Threshold')
ax2.axvline(n_components_90, color='#27ae60', linestyle=':', label=f'Chosen n={n_components_90}')
ax2.set_ylabel('Cumulative Explained Variance (%)', color='#d35400')
ax2.set_ylim(0, 105)

plt.title("PCA Scree Plot & Cumulative Explained Variance Ratio", weight='bold')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right')
plt.tight_layout()
plt.show()

# Train and evaluate SVM on the 6 PCA components
pca_opt = PCA(n_components=n_components_90, random_state=RANDOM_STATE)
X_tr_pca = pca_opt.fit_transform(X_train[all_features])
X_te_pca = pca_opt.transform(X_test[all_features])

svm_pca = SVC(C=best_svm.C, gamma=best_svm.gamma, kernel=best_svm.kernel, random_state=RANDOM_STATE)
svm_pca.fit(X_tr_pca, y_train)

svm_pca_tr = accuracy_score(y_train, svm_pca.predict(X_tr_pca))
svm_pca_te = accuracy_score(y_test, svm_pca.predict(X_te_pca))
svm_pca_cv = cross_val_score(svm_pca, X_tr_pca, y_train, cv=cv, scoring='accuracy')

print(f"SVM + PCA (n={n_components_90}) Test Acc: {svm_pca_te*100:.2f}%")
print(f"SVM + PCA (n={n_components_90}) CV Mean: {svm_pca_cv.mean()*100:.2f}% ± {svm_pca_cv.std()*100:.2f}%")


## Task 10: Final Results Table & Summarisation
Here we compile our final comparison table across all 4 SVM model configurations:
1. Baseline SVM (RBF, default parameters)
2. Tuned SVM (GridSearchCV: $C=100.0, \gamma=0.1$)
3. SVM with Feature Selection ($K=11$)
4. SVM with PCA Dimensionality Reduction ($n=6$)


In [10]:
final_models = {
    '1. Baseline SVM (RBF, default)': (svm_baseline, X_train[all_features], X_test[all_features], len(all_features), svm_base_tr, svm_base_te, svm_base_cv),
    '2. Tuned SVM (GridSearchCV)': (best_svm, X_train[all_features], X_test[all_features], len(all_features), svm_tuned_tr, svm_tuned_te, svm_tuned_cv),
    f'3. SVM + K-Best (K={best_k})': (svm_kbest, X_tr_kbest, X_te_kbest, best_k, svm_kbest_tr, svm_kbest_te, svm_kbest_cv),
    f'4. SVM + PCA (n={n_components_90})': (svm_pca, X_tr_pca, X_te_pca, n_components_90, svm_pca_tr, svm_pca_te, svm_pca_cv)
}

final_rows = []
for name, (mod, xtr, xte, nfeat, tr, te, cv_sc) in final_models.items():
    preds = mod.predict(xte)
    p, r, f, _ = precision_recall_fscore_support(y_test, preds, average='macro')
    final_rows.append({
        'Model Configuration': name,
        'Features': nfeat,
        'Train Acc (%)': tr * 100,
        'Test Acc (%)': te * 100,
        '5-Fold CV Mean (%)': cv_sc.mean() * 100,
        '5-Fold CV Std (%)': cv_sc.std() * 100,
        'Macro Precision (%)': p * 100,
        'Macro Recall (%)': r * 100,
        'Macro F1-Score (%)': f * 100
    })

final_table_df = pd.DataFrame(final_rows)
print("=== Final Results Table (Task 10) ===")
display(final_table_df)

# Plot comparison bar chart
fig, ax = plt.subplots(figsize=(10, 5), dpi=150)
x_pos = np.arange(len(final_table_df))
width = 0.35
ax.bar(x_pos - width/2, final_table_df['Test Acc (%)'], width, label='Test Accuracy (70/30 Split)', color='#2980b9', edgecolor='black')
ax.bar(x_pos + width/2, final_table_df['5-Fold CV Mean (%)'], width, yerr=final_table_df['5-Fold CV Std (%)'],
       capsize=5, label='5-Fold CV Accuracy (Mean ± Std)', color='#27ae60', edgecolor='black')

ax.set_xticks(x_pos)
ax.set_xticklabels(['Baseline SVM', 'Tuned SVM', f'SVM + K-Best\n(K={best_k})', f'SVM + PCA\n(n={n_components_90})'])
ax.set_ylabel("Accuracy (%)", weight='bold')
ax.set_title("Final Comparison of SVM Model Configurations", weight='bold')
ax.set_ylim(60, 95)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

# Confusion Matrix for Best Tuned SVM
cm = confusion_matrix(y_test, best_svm.predict(X_test[all_features]))
fig, ax = plt.subplots(figsize=(6, 5), dpi=150)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels, ax=ax)
ax.set_title(f"Confusion Matrix: Best Tuned SVM (C={best_svm.C}, γ={best_svm.gamma})", weight='bold')
ax.set_xlabel("Predicted Class")
ax.set_ylabel("True Class")
plt.tight_layout()
plt.show()


## Final Reflections & Recommendations

### 1. The Performance Journey
- **Original Decision Tree (Set 1):** 70.32% CV accuracy.
- **Engineered Decision Tree (Set 3):** 77.54% CV accuracy (+7.22% gain from domain feature engineering).
- **Baseline SVM (RBF default):** 78.78% CV accuracy.
- **Tuned SVM ($C=100.0, \gamma=0.1$):** 84.32% CV accuracy (+5.54% gain from tuning).
- **SVM with Feature Selection ($K=11$):** **84.60% CV accuracy** (Our best overall model).
- **SVM with PCA ($n=6$):** 81.83% CV accuracy (cut features in half with only ~2.5% accuracy trade-off).

### 2. Decision Trees vs SVM
Concrete hydration is continuous as calcium silicate hydrate gel forms. Decision trees try to approximate this smooth surface with blocky step functions, which introduces errors. SVM with an RBF kernel constructs smooth, curved decision boundaries with maximum margins, making it a much better match for the physical problem.

### 3. Feature Selection vs PCA
- **SelectKBest ($K=11$):** Keeps real physical measurements (kg/m³ of cement, sand, water), so civil engineers can audit and trust the predictions. It also achieved our highest accuracy (**84.60%**).
- **PCA ($n=6$):** Great for compression, cutting inputs from 12 to 6 while maintaining 81.83% accuracy. However, principal components are mathematical blends of features, so you lose the physical units.

### 4. Practical Recommendation for Civil Engineering
For a real construction site or batching plant following Australian Standard AS 3600, we recommend the **Tuned SVM with Feature Selection ($K=11$)**. It gives the highest accuracy while keeping the inputs fully explainable.
